# 🧊 Iceberg Superpowers — Schema, Partition & Sort Order Evolution

Features that were **impossible** (or incredibly painful) in older systems like Hadoop/Hive, delivered as instant metadata operations in Iceberg.

| Feature | Hive/Hadoop 😰 | Iceberg 🧊 |
|---------|---------------|-------------|
| Add a column | Rewrite all Parquet files | Metadata-only, instant |
| Rename a column | Not supported (positional reads) | Metadata-only, instant |
| Promote a type (INT→BIGINT) | Manual migration + rewrite | Safe promotions are metadata-only |
| Change partitioning | Create new table + full rewrite + migrate | Metadata-only, old data untouched |
| Change sort order | N/A | Metadata-only, future writes use new order |

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Trino

In [ ]:
from trino.dbapi import connect

conn = connect(
    host="trino",
    port=8080,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()


def run_query(sql, display=True):
    """Execute a query and return results as a formatted table."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [
                max(len(str(c)), max(len(str(r[i])) for r in rows))
                for i, c in enumerate(columns)
            ]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []


print("✅ Connected to Trino")

---
# Part 1 — Schema Evolution (Zero-Rewrite Magic)

In Hive, schemas are **coupled to the physical file layout**. Adding a column means rewriting every Parquet file. Renaming a column? Simply not possible — Hive uses positional column mapping, so column identity = column position.

Iceberg tracks columns by **unique IDs**, not positions or names. Every schema change is a metadata-only operation — zero data files are rewritten, regardless of table size.

## 1.1 — Create a Product Catalog Table

In [ ]:
import datetime

run_query("""
CREATE OR REPLACE TABLE iceberg.bronze.products (
    product_id   INTEGER,
    name         VARCHAR,
    price        REAL,
    category     VARCHAR
) WITH (format = 'PARQUET')
""")
print(f"✅ Table 'products' created at {table_created_at.strftime('%H:%M:%S UTC')}")
print("   product_id is INTEGER, price is REAL — intentionally narrow types for promotion demo")

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.products VALUES
    (1, 'Wireless Mouse',      29.99,  'Electronics'),
    (2, 'Mechanical Keyboard', 149.99, 'Electronics'),
    (3, 'Standing Desk',       549.00, 'Furniture'),
    (4, 'Monitor Arm',         89.50,  'Furniture'),
    (5, 'USB-C Hub',           45.00,  'Electronics')
""")
print("✅ 5 products inserted")

In [ ]:
print("📋 Original schema and data:")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

## 1.2 — Add a New Column

**In Hive:** You'd need to rewrite every Parquet file to add the column.

**In Iceberg:** One `ALTER TABLE` — instant, metadata-only. Existing rows return `NULL` for the new column.

In [ ]:
run_query("ALTER TABLE iceberg.bronze.products ADD COLUMN weight_kg DOUBLE")
print("✅ Column 'weight_kg' added — zero files rewritten!")

In [ ]:
print("📋 After ADD COLUMN — existing rows have NULL for weight_kg:")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

New inserts can populate the new column immediately:

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.products VALUES
    (6, 'Laptop Stand', 79.99, 'Furniture', 2.3)
""")
print("✅ New product inserted with weight_kg = 2.3")

## 1.3 — Rename a Column

**In Hive:** Impossible without breaking queries. Hive reads columns by **position**, not by name — renaming a column in the metastore would cause it to read the wrong data from existing files.

**In Iceberg:** Columns are tracked by **unique field IDs** embedded in the metadata. Renaming just updates the name → ID mapping. All existing data files remain valid.

In [ ]:
run_query("ALTER TABLE iceberg.bronze.products RENAME COLUMN category TO department")
print("✅ Column 'category' → 'department' — zero files rewritten!")

In [ ]:
print("📋 After RENAME COLUMN — 'category' is now 'department':")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

## 1.4 — Safe vs. Unsafe Type Promotion

Iceberg allows **widening** a column's type when it is guaranteed to be lossless — these are called *safe promotions*. Narrowing or incompatible changes are rejected outright.

| Promotion | Safe? | Why |
|-----------|:-----:|-----|
| `INTEGER` → `BIGINT` | ✅ Yes | Every 32-bit int fits in 64-bit |
| `REAL` → `DOUBLE` | ✅ Yes | Every 32-bit float fits in 64-bit |
| `REAL` → `DECIMAL` | ✅ Yes | Widening to exact numeric |
| `DOUBLE` → `INTEGER` | ❌ No | Narrowing — data loss possible |
| `VARCHAR` → `BIGINT` | ❌ No | Incompatible types |
| `BIGINT` → `INTEGER` | ❌ No | Narrowing — values > 2³¹ would overflow |

> **Why does this matter?** Writer schemas and reader schemas must stay compatible. A safe promotion means every existing Parquet value in the old type can be read as the new type without any file rewrite.

### ✅ Safe Promotion: `INTEGER` → `BIGINT` and `REAL` → `DOUBLE`

In [ ]:
# Current column types: product_id INTEGER, price REAL
print("Before promotion:")
run_query("""
SELECT column_name, data_type
FROM iceberg.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'products'
  AND column_name IN ('product_id', 'price')
ORDER BY column_name
""")

In [ ]:
# Promote INTEGER → BIGINT (safe widening)
run_query("""
ALTER TABLE iceberg.bronze.products
ALTER COLUMN product_id SET DATA TYPE BIGINT
""")

# Promote REAL → DOUBLE (safe widening)
run_query("""
ALTER TABLE iceberg.bronze.products
ALTER COLUMN price SET DATA TYPE DOUBLE
""")

print("✅ Safe promotions applied: INTEGER→BIGINT, REAL→DOUBLE — zero files rewritten!")

In [ ]:
print("After safe promotions:")
run_query("""
SELECT column_name, data_type
FROM iceberg.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'products'
  AND column_name IN ('product_id', 'price')
ORDER BY column_name
""")
print()
print("📋 Data still reads correctly with widened types:")
print()
run_query("SELECT * FROM iceberg.bronze.products ORDER BY product_id");

### ❌ Unsafe Promotion: `DOUBLE` → `INTEGER` (Rejected)

Attempting to narrow a column's type is always blocked by Iceberg — it would be impossible to guarantee correctness for existing data files that contain the wider type.

In [ ]:
import traceback

try:
    # UNSAFE: narrowing DOUBLE → INTEGER loses decimal precision
    cursor.execute("""
        ALTER TABLE iceberg.bronze.products
        ALTER COLUMN price SET DATA TYPE INTEGER
    """)
    cursor.fetchall()
    print("⚠️  Unexpected: no error raised!")
except Exception as e:
    print("❌ Unsafe promotion correctly rejected by Iceberg:")
    print(f"   {type(e).__name__}: {str(e)[:200]}")

---
# Part 2 — Partition Evolution (Hidden Partitioning)

In Hive, partitioning is **baked into the directory structure** (`/year=2026/month=02/day=15/`). Changing the partition scheme means:
1. Create a brand-new table with the new partitioning
2. Copy (rewrite) all historical data
3. Migrate all downstream queries
4. Drop the old table

In Iceberg, partition specs are **tracked in metadata** and completely hidden from queries. You can evolve the partitioning at any time — old data keeps its original layout, new data uses the new scheme, and both are queried transparently.

```
                  Partition Evolution Timeline
──────────────────────────────────────────────────────────────
  Old data (spec v0)              New data (spec v1)
  Partitioned by MONTH            Partitioned by DAY
  ┌─────────┐                     ┌──────────────┐
  │ 2026-01 │                     │ 2026-03-01   │
  │ 2026-02 │  ── evolve ──▶      │ 2026-03-02   │
  │ 2026-03 │                     │ 2026-03-03   │
  └─────────┘                     └──────────────┘
        Both coexist — Iceberg handles it transparently
```

## 2.1 — Create an Event Table Partitioned by MONTH

In [ ]:
run_query("DROP TABLE IF EXISTS iceberg.bronze.events")

run_query("""
CREATE TABLE iceberg.bronze.events (
    event_id   BIGINT,
    user_id    BIGINT,
    event_type VARCHAR,
    event_ts   TIMESTAMP(6) WITH TIME ZONE
) WITH (
    format = 'PARQUET',
    partitioning = ARRAY['month(event_ts)']
)
""")
print("✅ Table 'events' created — partitioned by MONTH(event_ts)")

### 📥 Load Historical Data (Jan–Feb 2026)

Iceberg automatically routes each row to the correct monthly partition — no explicit partition column needed in the INSERT.

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.events VALUES
    (1,  101, 'page_view',   TIMESTAMP '2026-01-05 10:00:00.000000 UTC'),
    (2,  102, 'click',       TIMESTAMP '2026-01-12 14:30:00.000000 UTC'),
    (3,  101, 'purchase',    TIMESTAMP '2026-01-20 09:15:00.000000 UTC'),
    (4,  103, 'page_view',   TIMESTAMP '2026-02-03 11:00:00.000000 UTC'),
    (5,  101, 'click',       TIMESTAMP '2026-02-14 16:45:00.000000 UTC'),
    (6,  104, 'signup',      TIMESTAMP '2026-02-28 08:30:00.000000 UTC'),
    (7,  102, 'page_view',   TIMESTAMP '2026-01-08 12:00:00.000000 UTC'),
    (8,  105, 'purchase',    TIMESTAMP '2026-02-10 15:20:00.000000 UTC'),
    (9,  103, 'click',       TIMESTAMP '2026-01-25 13:10:00.000000 UTC'),
    (10, 104, 'page_view',   TIMESTAMP '2026-02-18 10:05:00.000000 UTC')
""")
print("✅ 10 events inserted across Jan-Feb 2026")

### 🔍 Inspect the Physical Partitioning

The `$partitions` metadata table shows how data is physically organized. Currently, data is grouped into monthly buckets.

In [ ]:
print("📁 Partition layout (MONTH granularity):")
print()
run_query("""
SELECT
    partition.event_ts_month AS partition_value,
    record_count,
    file_count
FROM iceberg.bronze."events$partitions"
""");

## 2.2 — Evolve Partition Spec: MONTH → DAY

**In Hive:** This is a nightmare — full table rewrite required.

**In Iceberg:** One DDL statement. Existing data **stays in monthly partitions**. Only new writes use daily partitions. Iceberg's query planner handles both transparently.

In [ ]:
run_query("""
ALTER TABLE iceberg.bronze.events
SET PROPERTIES partitioning = ARRAY['day(event_ts)']
""")
print("✅ Partition spec evolved: MONTH(event_ts) → DAY(event_ts)")
print("   Old data is untouched. New writes will use daily partitions.")

### 📥 Insert New Data (Mar 2026)

These new events will be written into **daily** partitions — completely transparent to the INSERT.

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.events VALUES
    (11, 101, 'page_view', TIMESTAMP '2026-03-01 09:00:00.000000 UTC'),
    (12, 102, 'click',     TIMESTAMP '2026-03-01 14:20:00.000000 UTC'),
    (13, 103, 'purchase',  TIMESTAMP '2026-03-02 11:30:00.000000 UTC'),
    (14, 105, 'signup',    TIMESTAMP '2026-03-02 16:00:00.000000 UTC'),
    (15, 101, 'click',     TIMESTAMP '2026-03-03 08:45:00.000000 UTC'),
    (16, 104, 'page_view', TIMESTAMP '2026-03-03 13:15:00.000000 UTC'),
    (17, 102, 'purchase',  TIMESTAMP '2026-03-04 10:00:00.000000 UTC'),
    (18, 106, 'signup',    TIMESTAMP '2026-03-04 15:30:00.000000 UTC')
""")
print("✅ 8 new events inserted (Mar 1-4) — using DAY partitions")

### 🔍 Inspect the Mixed Partition Layout

Now the table has **two partition specs coexisting**:
- Old data → monthly partitions (spec v0)
- New data → daily partitions (spec v1)

We can see this by looking at the data files and their partitions:

In [ ]:
print("📁 Data files — showing mixed monthly + daily partitions:")
print()
run_query("""
SELECT
    partition,
    record_count,
    file_size_in_bytes
FROM iceberg.bronze."events$files"
""");

### ✅ Queries Work Transparently Across Both Specs

The key insight: **queries don't know or care about the partition layout**. Iceberg's planner reads from both monthly and daily partitions seamlessly.

In [ ]:
print("📋 All events across ALL partition specs — seamless query:")
print()
run_query("SELECT * FROM iceberg.bronze.events ORDER BY event_ts");

In [ ]:
print("📊 Events per day — aggregation works across both partition specs:")
print()
run_query("""
SELECT
    CAST(event_ts AS DATE) AS event_date,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_id) AS unique_users
FROM iceberg.bronze.events
GROUP BY CAST(event_ts AS DATE)
ORDER BY event_date
""");

---
# Part 3 — Sort Order Evolution

Iceberg tracks a **sort order** in table metadata, separate from the schema and partition spec. This sort order is used by writers (e.g. compaction jobs) to physically arrange rows within each data file, enabling better compression and faster range queries.

| Aspect | Details |
|--------|---------|
| **Scope** | Per-file row ordering, not global |
| **Effect** | Writers use the current sort order; existing files are untouched |
| **Query impact** | Engines can push down range predicates more efficiently |
| **Cost** | Metadata-only to *change*; compaction rewrites files to *apply* |

```
         Sort Order Evolution — the before/after view
  ──────────────────────────────────────────────────────────
  Existing files (unsorted or old order)     New files
  ┌──────────────────────────────────┐       ┌──────────────────────────┐
  │ rows in arrival order            │       │ rows sorted by           │
  │ (or sorted by old spec)          │  ──▶  │ order_ts ASC,            │
  │                                  │       │ customer_id ASC          │
  └──────────────────────────────────┘       └──────────────────────────┘
    Untouched by the ALTER                    Written by future INSERTs
```

## 3.1 — Create an Orders Table (No Sort Order)

In [ ]:
run_query("DROP TABLE IF EXISTS iceberg.bronze.orders")

run_query("""
CREATE TABLE iceberg.bronze.orders (
    order_id    BIGINT,
    customer_id BIGINT,
    region      VARCHAR,
    amount      DOUBLE,
    order_ts    TIMESTAMP(6) WITH TIME ZONE
) WITH (
    format       = 'PARQUET',
    partitioning = ARRAY['month(order_ts)']
)
""")
print("✅ Table 'orders' created — no sort order yet")

In [ ]:
# Insert rows in random/arrival order — no sorting applied
run_query("""
INSERT INTO iceberg.bronze.orders VALUES
    (1005, 302, 'APAC',   340.00, TIMESTAMP '2026-01-15 08:00:00.000000 UTC'),
    (1001, 101, 'EMEA',   250.00, TIMESTAMP '2026-01-03 09:00:00.000000 UTC'),
    (1003, 202, 'AMER',   180.50, TIMESTAMP '2026-01-10 11:00:00.000000 UTC'),
    (1002, 101, 'AMER',   520.75, TIMESTAMP '2026-01-07 14:00:00.000000 UTC'),
    (1004, 303, 'EMEA',    95.00, TIMESTAMP '2026-01-12 16:00:00.000000 UTC')
""")
print("✅ 5 orders inserted in random order")
print()
print("📋 Rows as stored (arrival / heap order):")
print()
run_query("SELECT * FROM iceberg.bronze.orders");

## 3.2 — Set a Sort Order on the Table

We evolve the table to declare that future writes should be **sorted by `order_ts` ASC** (then by `customer_id` ASC as a tiebreaker). This is a metadata-only operation — the existing unsorted file is untouched.

In [ ]:
run_query("""
ALTER TABLE iceberg.bronze.orders
SET PROPERTIES sorted_by = ARRAY['order_ts ASC', 'customer_id ASC']
""")
print("✅ Sort order set: order_ts ASC, customer_id ASC")
print("   Existing data files are NOT rewritten — sort takes effect on future writes.")

In [ ]:
# Verify the new sort order is recorded in table properties
print("📋 Table properties — confirming sort order is recorded:")
print()
run_query("""
SELECT key, value
FROM iceberg.bronze."orders$properties"
WHERE key LIKE '%sort%'
""");

## 3.3 — New Inserts Respect the Sort Order

Insert a new batch for February in deliberately jumbled order. A writer that honours the table sort order will **sort these rows before flushing to Parquet**, producing a file where rows are ordered by `order_ts` then `customer_id`.

In [ ]:
run_query("""
INSERT INTO iceberg.bronze.orders VALUES
    (1010, 202, 'APAC',   610.00, TIMESTAMP '2026-02-18 10:00:00.000000 UTC'),
    (1007, 101, 'EMEA',   430.00, TIMESTAMP '2026-02-07 09:00:00.000000 UTC'),
    (1009, 303, 'AMER',   275.00, TIMESTAMP '2026-02-14 08:30:00.000000 UTC'),
    (1006, 101, 'AMER',   190.00, TIMESTAMP '2026-02-02 11:00:00.000000 UTC'),
    (1008, 202, 'EMEA',   380.00, TIMESTAMP '2026-02-10 14:00:00.000000 UTC')
""")
print("✅ 5 orders inserted for Feb 2026 (jumbled input order)")

In [ ]:
print("📋 Data files — note the two files and their row ordering:")
print()
run_query("""
SELECT
    partition,
    record_count,
    column_sizes
FROM iceberg.bronze."orders$files"
ORDER BY partition
""");

## 3.4 — Evolve the Sort Order: Add `region` as Primary Sort Key

Assume queries now frequently filter on `region` first, so we want files sorted by `region` first. We issue a second `ALTER TABLE` — again metadata-only. All existing files (sorted by `order_ts` or unsorted) are untouched.

In [ ]:
run_query("""
ALTER TABLE iceberg.bronze.orders
SET PROPERTIES sorted_by = ARRAY['region ASC', 'order_ts ASC']
""")
print("✅ Sort order evolved: now region ASC, order_ts ASC")
print("   Existing files keep their old ordering — only future writes use the new spec.")

In [ ]:
# New write — rows within the March file will be sorted by region then order_ts
run_query("""
INSERT INTO iceberg.bronze.orders VALUES
    (1015, 101, 'EMEA',   770.00, TIMESTAMP '2026-03-15 09:00:00.000000 UTC'),
    (1011, 303, 'AMER',   310.00, TIMESTAMP '2026-03-03 10:00:00.000000 UTC'),
    (1013, 101, 'AMER',   450.00, TIMESTAMP '2026-03-09 11:00:00.000000 UTC'),
    (1014, 202, 'APAC',   620.00, TIMESTAMP '2026-03-12 13:00:00.000000 UTC'),
    (1012, 303, 'EMEA',   280.00, TIMESTAMP '2026-03-06 15:00:00.000000 UTC')
""")
print("✅ 5 orders inserted for Mar 2026 — sorted by region ASC, order_ts ASC")
print()
print("📋 All orders — three files with different physical sort orders coexist:")
print()
run_query("SELECT * FROM iceberg.bronze.orders ORDER BY order_ts");

---
## 📊 Final Summary — All Three Evolution Types

| Evolution Type | Operation | SQL Verb | Data Rewrite? | When Applied? |
|----------------|-----------|----------|:---:|---------------|
| **Schema** | Add column | `ADD COLUMN` | ❌ | Immediately, old rows return NULL |
| **Schema** | Rename column | `RENAME COLUMN` | ❌ | Immediately, all files remain valid |
| **Schema** | Safe type promotion (INT→BIGINT, REAL→DOUBLE) | `SET DATA TYPE` | ❌ | Immediately, lossless widening |
| **Schema** | Unsafe type change (DOUBLE→INT) | *(rejected)* | — | Blocked by Iceberg |
| **Partition** | Change partition transforms | `SET PROPERTIES partitioning` | ❌ | New writes only; old files keep old spec |
| **Sort Order** | Set / change sort key | `SET PROPERTIES sorted_by` | ❌ | New writes only; old files untouched |

> **Key insight:** All *evolution* operations are metadata-only and instantaneous. The cost of physically reorganising data (sorting, repartitioning) is paid **lazily** during compaction, not at DDL time.

---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop all tables:
# run_query("DROP TABLE IF EXISTS iceberg.bronze.products")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.events")
# run_query("DROP TABLE IF EXISTS iceberg.bronze.orders")
# print("🗑️ All tables dropped")